# Assignment: Extend the az.ipynb Lab

**Based on:** `Lab2.ipynb` (the Module 3 lab).

This week's assignment is short on purpose: take your working `Lab2.ipynb` lab and add **one
more step** to the chain. No new concepts, no new setup, no new libraries — just one more
chained LLM call that builds on what you already have.

**Two deployments this time:** `gpt-5.1-ptu` is the default deployment for every existing
step (Steps 1–4). The new step you add (Step 5) must call `gpt-5.4-ptu` instead.

## Step 1 — Start from your working lab

- Make a copy of your completed `Lab2.ipynb` (e.g. rename the copy `assignment3.ipynb`), or
  continue directly inside this notebook — either is fine.
- Copy in your working code from the lab's Steps 1–4: the imports and `.env` config, the
  `AzureOpenAI` client, the `chat()` helper, and the chain itself (fun fact → generate a hard
  question → answer it → evaluate the answer).
- Confirm your `.env`'s `AZURE_APIM_OPENAI_DEPLOYMENT` is set to `gpt-5.1-ptu` — this stays
  the default deployment for Steps 1–4, unchanged.

Run those cells first and confirm they still work before moving on.

In [1]:
from dotenv import load_dotenv
import os
import sys

from openai import AzureOpenAI

# Read .env and override any existing process env values.
load_dotenv(override=True)

# APIM settings from .env. Endpoint must be the host only, e.g.
# https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net  (no /openai/deployments)
api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_APIM_OPENAI_API_VERSION")
endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT")

if not all([api_key, api_version, endpoint, deployment]):
    sys.exit(
        "Missing Azure APIM settings. Set AZURE_APIM_OPENAI_SUBSCRIPTION_KEY, "
        "AZURE_APIM_OPENAI_API_VERSION, AZURE_APIM_OPENAI_ENDPOINT, and "
        "AZURE_APIM_OPENAI_DEPLOYMENT in your .env file."
    )

print(f"Azure APIM key exists and begins {api_key[:8]}")
print(f"Deployment: {deployment}")

# Sync client pointed at Azure APIM.
client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

# On Azure the `model` argument is the *deployment name*, not an OpenAI model id.
# GPT-5 deployments need max_completion_tokens (max_tokens is rejected).
def chat(messages, max_completion_tokens=1000, deployment=deployment):
    return client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_completion_tokens=max_completion_tokens,
    )

    # 1) Fun fact
messages = [{"role": "user", "content": "Tell me a short fun fact"}]
response = chat(messages)
print(response.choices[0].message.content)

# 2) Ask the model to invent a hard IQ-style question
question = (
    "Please propose a hard, challenging question to assess someone's IQ. "
    "Respond only with the question."
)
messages = [{"role": "user", "content": question}]
response = chat(messages)
question = response.choices[0].message.content
print(question)

# 3) Ask the model to answer that question
messages = [{"role": "user", "content": question}]
response = chat(messages)
answer = response.choices[0].message.content
print(answer)

# Render the answer as Markdown in the notebook.
from IPython.display import Markdown, display

display(Markdown(answer))

# 4) Ask the model to evaluate the answer
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""
print(message)

messages = [{"role": "user", "content": message}]
response = chat(messages, max_completion_tokens=5000)
print(response.choices[0].message.content)



Azure APIM key exists and begins b5309e0c
Deployment: gpt-5.1-ptu
Octopuses have three hearts: two pump blood to the gills, and one pumps it to the rest of the body—and the main heart actually stops beating when they swim, which is part of why they prefer to crawl instead of swim.
A clock is set correctly at 12:00 noon. It loses exactly 3 minutes every hour. After how many hours will the clock first show the correct time again (though actually being wrong), and what real time will it be then?
The clock loses 3 minutes every real hour, so it runs at:

- 60 − 3 = 57 minutes of *its* time per 60 minutes of real time  
- Rate = 57/60 = 19/20 of real-time speed

We want the first time (after 12:00) when its *displayed* time matches the real time again.

Let \(t\) = real time elapsed in minutes.  
The clock’s reading (in minutes past 12:00) is \(\frac{19}{20}t\).

They show the same time when the difference is a whole number of 12-hour cycles (720 minutes):

\[
t - \frac{19}{20}t = 720
\]

\

The clock loses 3 minutes every real hour, so it runs at:

- 60 − 3 = 57 minutes of *its* time per 60 minutes of real time  
- Rate = 57/60 = 19/20 of real-time speed

We want the first time (after 12:00) when its *displayed* time matches the real time again.

Let \(t\) = real time elapsed in minutes.  
The clock’s reading (in minutes past 12:00) is \(\frac{19}{20}t\).

They show the same time when the difference is a whole number of 12-hour cycles (720 minutes):

\[
t - \frac{19}{20}t = 720
\]

\[
\frac{1}{20}t = 720
\]

\[
t = 14400 \text{ minutes}
\]

Convert 14,400 minutes:

\[
14400 \div 60 = 240 \text{ hours} = 10 \text{ days}
\]

So:

- It first shows the correct time again after **240 hours** of real time.  
- The real time then will be **12:00 noon, 10 days later**.


Here is a question:
A clock is set correctly at 12:00 noon. It loses exactly 3 minutes every hour. After how many hours will the clock first show the correct time again (though actually being wrong), and what real time will it be then?

And here is a possible answer that might be correct or incorrect:
The clock loses 3 minutes every real hour, so it runs at:

- 60 − 3 = 57 minutes of *its* time per 60 minutes of real time  
- Rate = 57/60 = 19/20 of real-time speed

We want the first time (after 12:00) when its *displayed* time matches the real time again.

Let \(t\) = real time elapsed in minutes.  
The clock’s reading (in minutes past 12:00) is \(\frac{19}{20}t\).

They show the same time when the difference is a whole number of 12-hour cycles (720 minutes):

\[
t - \frac{19}{20}t = 720
\]

\[
\frac{1}{20}t = 720
\]

\[
t = 14400 \text{ minutes}
\]

Convert 14,400 minutes:

\[
14400 \div 60 = 240 \text{ hours} = 10 \text{ days}
\]

So:

- It first shows the correct time again after 

## Step 2 - Add one more chained step

In [2]:
# Step 5 - rate the difficulty of the question

prompt = (
    f"Here is a question: \n{question}\n\n"
    f"Here is the answer given:\n{answer}\n\n"
    "On a scale of 1 to 10, how difficult is this question? "
    "Give the number and one-sentence justification."
)
messages = [{"role": "user", "content": prompt}]
response = chat(messages, deployment="gpt-5.4-ptu")
difficulty_rating = response.choices[0].message.content

print("=== STEP 5 (gpt-5.4-ptu) ===")
print(difficulty_rating)

=== STEP 5 (gpt-5.4-ptu) ===
4 — It’s a fairly straightforward rate-and-clock-cycle problem once you realize the clock must be 12 hours behind to look correct again.


## Reflection

Answer in a sentence or two each:

1. **Which earlier variable(s) did your Step 5 prompt use, and why that one?**
2. **What would break if you ran Step 5 before the step it depends on?**
3. **Why might a real project deliberately use a different deployment (e.g. a stronger or
   more expensive model) for just one step in a chain, instead of using it everywhere?**

### My reflection

1. I used both question and answer in my prompt. I chose those two because I think rating the difficulty of something depends on how well you know something that was asked of you and what you'd give as a response. 
2. I think if my question and answer variables had not existed yet then I would have gotten a NameError output because Step 5 relies on Steps 2 and 3 occurring. 
3. I think a real project would use a different deployment such as a stronger one because if the question/answer requires more careful reasoning than answering it quickly and a weaker model might create worse results.

## Submission checklist

- [*] Notebook runs top to bottom without errors (`Kernel → Restart & Run All`)
- [ ] `.env` file is **not** included in your submission
- [*] Step 5 is clearly labeled and its prompt uses at least one earlier variable
- [*] Reflection questions are answered